# Milestone 5 lab, notebook 2 of 8: On time, as a number

## Where we are

| Notebook | What you do |
|---|---|
| 1. Did it run? | Read Airflow's run history and the pipeline's own. Run one night twice |
| **2. On time, as a number** (you are here) | Measure freshness. Turn "on time" into a number with a target |
| 3. Five signals | Monitoring asks if the job ran. Observability asks if the data is right |
| 4. Tell the person who can act | Airflow calls back, and the right person gets an alert |
| 5. Too many alerts | Count the careless way. Then group, send once, rank and silence |
| 6. Fix one day | Fix the source, run one night again, prove the number |
| 7. Four drills | Four faults on a clean day, each caught and repaired. Write an incident note |
| 8. The Monday answer | The whole week, the recovery, and the answer to the question |

**You need:** the warehouse and Airflow running. From the project folder: `docker compose up -d --build`. Airflow's page is http://localhost:8080 .

**About ten minutes of reading, and two minutes of waiting** while Airflow runs the last four nights of the week.

In [1]:
import pandas as pd
from lab_helpers import (
    FRI, SAT, SUN, THU, WED, gate_results, prepare_nights, run_nights, show_file, show_health, sql,
    store_day,
)

prepare_nights(WED)

The bonus week ran under Airflow: 3 nights, Mon 08-24 to Wed 08-26. 4 of 6 stores can be paid.


## Step 1: When is a file on time?

**Aiko Tanaka** runs store `S06`. On Thursday 27 August her store traded as usual. Her sales file arrived **23.7 hours** after it was due. In this notebook you watch the pipeline notice it, and then you turn "on time" into a number.

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, what happens when it fails, and who is told.

First, the promise. Lotus wrote it down in this milestone's BRD:

| Promise | The number |
|---|---|
| A store's sales file arrives | By **01:00** the night after the business day |
| The pipeline starts | At **02:00** |

Here is when the six files of Wednesday arrived:

In [2]:
sql("""
    SELECT file_name, received_at
    FROM raw.loaded_files
    WHERE table_name = 'sales_lines' AND file_name LIKE '%2026-08-26%'
    ORDER BY file_name
""")

,file_name,received_at
0,sales_lines_S01_2026-08-26.csv,2026-08-26 23:10:00
1,sales_lines_S02_2026-08-26.csv,2026-08-26 23:14:00
2,sales_lines_S03_2026-08-26.csv,2026-08-26 23:18:00
3,sales_lines_S04_2026-08-26.csv,2026-08-26 23:22:00
4,sales_lines_S05_2026-08-26.csv,2026-08-26 23:26:00
5,sales_lines_S06_2026-08-26.csv,2026-08-26 23:30:00


All six before midnight. All on time.

**Freshness** is how new the data is. A freshness check asks: did the newest data arrive in time? Here that means: did every sales file arrive by 01:00?

## Step 2: The freshness test

The test is a dbt test at the landing gate. It returns every file that arrived after it was due.

In [3]:
show_file("dbt_project/tests/landing/file_arrived_on_time.sql", "SELECT", 12)

# dbt_project/tests/landing/file_arrived_on_time.sql, from line 12
SELECT file_name, substr(file_name, 13, 3) AS store_id, file_day AS sales_date, received_at,
       CAST(file_day + 1 AS TIMESTAMP) + INTERVAL '1 hour' AS due_at
FROM (
    -- Only a store's own files carry their day at this place in the name.
    SELECT file_name, received_at,
           CASE WHEN table_name IN ('sales_lines', 'till_totals')
                THEN CAST(substr(file_name, 17, 10) AS DATE) END AS file_day
    FROM {{ source('raw', 'loaded_files') }}
) files
WHERE file_day IS NOT NULL
  AND received_at > CAST(file_day + 1 AS TIMESTAMP) + INTERVAL '1 hour'


Read the `WHERE` line: a file is late when `received_at` is after 01:00 of the next day. Its action is **warn**: a late file has arrived, its rows are good, so it is loaded.

## Step 3: Thursday night

Let Airflow run Thursday.

In [4]:
run_nights([THU])

,night,tasks,succeeded,seconds,stores that can be paid
0,Thu 08-27,8,8,24,2


In [5]:
thursday = gate_results(THU, only_failed=False)
thursday[thursday["test_name"].isin(["file_arrived_on_time", "store_is_not_ready_to_pay"])]

,gate,test_name,action,status,bad_rows,title
3,landing,file_arrived_on_time,warn,pass,0,A store's file arrives before 01:00
45,publish,store_is_not_ready_to_pay,warn,fail,4,Every open store has a day we can pay from


In [6]:
store_day("S06", THU)

,store_id,sales_date,lines_cents,till_cents,rows_set_aside,status,problem
0,S06,2026-08-27,0,61644,0,do not pay yet,no_sales_file


`S06` has no sales file for Thursday. And the freshness test says `pass`.

That is not a bug. The test looks at the files that arrived, and asks if each one was late. Aiko's file did not arrive at all. A row that is not there cannot be late.

The other test caught it: "Every open store has a day we can pay from", at the publish gate. It knows which stores were open, so it sees a store that sent nothing.

## Step 4: Friday night

Aiko's Thursday file arrives at 00:40 on Saturday 29 August: just before Friday's night run at 02:00. Run Friday's night.

In [7]:
run_nights([FRI])

,night,tasks,succeeded,seconds,stores that can be paid
0,Fri 08-28,8,8,19,3


In [8]:
friday = gate_results(FRI, only_failed=False)
friday[friday["test_name"].isin(["file_arrived_on_time", "store_is_not_ready_to_pay"])]

,gate,test_name,action,status,bad_rows,title
3,landing,file_arrived_on_time,warn,fail,1,A store's file arrives before 01:00
45,publish,store_is_not_ready_to_pay,warn,fail,3,Every open store has a day we can pay from


In [9]:
sql("""
    SELECT file_name, received_at, due_at,
           round(CAST(extract(epoch FROM received_at - due_at) / 3600 AS NUMERIC), 1) AS hours_late
    FROM audit.file_arrived_on_time
    WHERE file_name LIKE 'sales_lines%'
""")

,file_name,received_at,due_at,hours_late
0,sales_lines_S06_2026-08-27.csv,2026-08-29 00:40:00,2026-08-28 01:00:00,23.7


Now the freshness test fails, with one row: Aiko's file, **23.7 hours late**. And the store day passes: the file is here, loaded, and it adds up.

The two checks share the work:

| | Thursday night: the file is not there | Friday night: it has arrived |
|---|---|---|
| Every open store has a day we can pay from | **Fails**: no sales file arrived | Passes |
| A store's file arrives before 01:00 (freshness) | Passes: nothing to judge | **Fails**: 23.7 hours late |

A freshness check alone would have stayed silent on Thursday night, the one night it mattered.

## Step 5: dbt has a freshness command. Why not use it?

dbt has its own command, `dbt source freshness`: it finds the newest row of a table and compares it with the clock on the wall. That is right for a live system, where "now" is now.

This project's data is from August 2026, and you are running it on another day. dbt's command would call every file months late. So this test compares each file with **the night's own clock**: 01:00 after the day in the file's name.

## Step 6: "On time", as a number

Run the last two nights of the week.

In [10]:
run_nights([SAT, SUN])

,night,tasks,succeeded,seconds,stores that can be paid
0,Sat 08-29,8,8,24,2
1,Sun 08-30,8,8,24,1


Now ask four plain questions about the whole week, each answered with a number. This is the command `uv run python -m lotus_pipeline health --day 2026-08-30`:

In [11]:
show_health(SUN)

Is the pipeline healthy? Week of Mon 2026-08-24, up to Sun 2026-08-30

  ok   Did it run?              7 of 7 nights published
  ok   Did it finish on time?   the slowest night took 5 seconds. Target: under 300
  MISS Is the data fresh?       39 of 41 sales files arrived by 01:00: 95.1%. Target: 98%
         late:    S06 2026-08-27, 23.7 hours after it was due
         missing: S04 2026-08-25
       Did every store report?  36 of 41 store days can be paid from: 87.8%


Three words for what you see.

- An **SLI** (service level indicator) is a number you measure. "39 of 41 sales files arrived by 01:00" is an SLI.
- An **SLO** (service level objective) is the target for it. "98%" is the SLO.
- An **SLA** (service level agreement) is a promise to other people, with a cost when it is broken.

| | SLI | SLO | SLA |
|---|---|---|---|
| What it is | A number you measure | Your target for that number | A promise to others, with a cost |
| Lotus | 39 of 41 files on time: 95.1% | 98% | None in writing |
| Who sets it | It is measured | The data team | The business, with the people it serves |

The SLO was missed: 95.1% is less than 98%. Two files: `S06`'s Thursday file was late, and `S04`'s Tuesday file is missing. "The files are usually on time" was an opinion. "95.1% against 98%" is a miss that can be discussed.

Why 41 files and not 42? Six stores, seven days, less one: `S01` was closed on Thursday for a planned refit. A closed store sends nothing, and that is correct.

Lotus has no SLA. If the head of retail operations promised the managers "your bonus is confirmed by Monday 09:00, or you are told why", that would be one, and the next notebooks would be how she keeps it.

The last line, "Did every store report?", has no target. 36 of 41 store days can be paid from. A person judges that number: it is shown, not graded.

## What you now know

- **Freshness** is how new the data is. Here: did each sales file arrive by 01:00?
- A freshness check sees a late file only once it has arrived. A missing file needs a second check: did every open store report?
- dbt's `source freshness` compares with today's clock. This project compares with the night's own clock, because the data is from August 2026.
- An **SLI** is a number you measure, an **SLO** is its target, an **SLA** is a promise to others with a cost.

**Say it in an interview:** "Store files were due by 01:00. My SLI was the share that arrived by then, and the SLO was 98%. In the bonus week it was 95.1%: 39 of 41. One file was 23.7 hours late, and my freshness test named it once it arrived. It could not see the file that never came, so a second check asked whether every open store reported."

**The theory:** {OBS}, {DBT_FRESH}, and for the SLA: {AF_BACKFILL}.

Next: [notebook 3](03_five_signals.ipynb). Freshness is one signal. There are four more.